In [2]:
!pip install selenium
!apt-get update # to update ubuntu to correctly run apt install
!apt install chromium-chromedriver
!cp /usr/lib/chromium-browser/chromedriver /usr/bin
import sys


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 40.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 467.7/467.7 kB 38.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 5.9 MB/s eta 0:00:00
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [968 kB]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/c2d4u.team/c2d4u4.0+/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:10 http://security.ubun

In [3]:
import csv
import os
import time
from datetime import date
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException

# Inicialize o WebDriver
opções = webdriver.ChromeOptions()
opções.add_argument('--headless')
opções.add_argument('--no-sandbox')
opções.add_argument('--disable-dev-shm-usage')

driver = webdriver.Chrome(options=opções)

# Função para navegar até uma página específica
def navegar_para_pagina(pagina):
    url = f'https://www.joli.com.br/chuveiros-e-aquecedores?page={pagina}'
    driver.get(url)

# Criar pasta de saída se não existir
output_folder = "output"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"\033[92mA pasta {output_folder} foi criada.\033[0m")

# Encontre os produtos
produtos = []
produtos_carregados = set()  # Usaremos um set para rastrear produtos únicos
pagina_atual = 1
tempo_espera_inicial = 2
incremento_tempo_espera = 2
tempo_espera_maximo = 10

while True:
    try:
        # Navegue até a página atual
        navegar_para_pagina(pagina_atual)

        # Aguarde até que os produtos sejam carregados
        produtos_atual = WebDriverWait(driver, tempo_espera_inicial).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, '.vtex-search-result-3-x-galleryItem'))
        )

        # Adicione os produtos à lista
        novos_produtos = False
        for produto in produtos_atual:
            try:
                # Coletar informações do produto
                nome_empresa = "Joli"
                data_coleta = date.today().strftime("%Y-%m-%d")
                codigo_produto_element = produto.find_element(By.CSS_SELECTOR, '.vtex-product-summary-2-x-brandName')
                codigo_produto = codigo_produto_element.get_attribute('textContent').strip() if codigo_produto_element else "0"
                imagem_element = produto.find_element(By.CSS_SELECTOR, '.vtex-product-summary-2-x-image')
                imagem_url = imagem_element.get_attribute('src')
                nome_produto_element = produto.find_element(By.CSS_SELECTOR, '.vtex-product-summary-2-x-productNameContainer')
                nome_produto = nome_produto_element.get_attribute('textContent').strip()
                moeda_element = produto.find_element(By.CSS_SELECTOR, '.joli-joli-theme-0-x-price')
                moeda = moeda_element.get_attribute('textContent').strip()
                url_produto = produto.find_element(By.CSS_SELECTOR, 'a').get_attribute('href')

                # Montar a linha de dados do produto
                linha_produto = [nome_empresa, data_coleta, codigo_produto, imagem_url, nome_produto, moeda, url_produto]

                # Adicionar à lista de produtos
                produtos.append(linha_produto)
                produtos_carregados.add(nome_produto)  # Usamos nome_produto como identificador único

                novos_produtos = True
            except StaleElementReferenceException:
                # Se o elemento não estiver mais presente, ignore-o
                continue

        print(f"Total de produtos coletados até agora: {len(produtos)}")

        if not novos_produtos:
            print("Não há mais novos produtos para carregar.")
            break

        # Aumentar o tempo de espera para a próxima iteração
        tempo_espera_inicial += incremento_tempo_espera
        if tempo_espera_inicial > tempo_espera_maximo:
            tempo_espera_inicial = tempo_espera_maximo

        # Avance para a próxima página
        pagina_atual += 1

    except TimeoutException:
        # Se não encontrar mais produtos, saia do loop
        print("Não foi possível encontrar mais produtos.")
        break

# Salvar os produtos em um arquivo CSV
nome_arquivo = os.path.join(output_folder, 'chuveiros-e-aquecedores.csv')
with open(nome_arquivo, 'w', newline='', encoding='utf-8') as arquivo_csv:
    escritor_csv = csv.writer(arquivo_csv)
    escritor_csv.writerow(['Nome empresa', 'Data de coleta', 'Código do produto', 'Imagem', 'Nome do produto', 'Moeda', 'URL do produto'])

    for produto in produtos:
        escritor_csv.writerow(produto)

print(f"Os produtos foram salvos em: {nome_arquivo}")

# Feche o driver
driver.quit()


A pasta output foi criada.
Total de produtos coletados até agora: 1
Total de produtos coletados até agora: 5
Total de produtos coletados até agora: 10
Total de produtos coletados até agora: 13
Total de produtos coletados até agora: 15
Total de produtos coletados até agora: 21
Total de produtos coletados até agora: 26
Total de produtos coletados até agora: 26
Não há mais novos produtos para carregar.
Os produtos foram salvos em: output/chuveiros-e-aquecedores.csv
